# Phase 1 — Exploratory Data Analysis
NASA CMAPSS Turbofan Engine Degradation Dataset

**Goals:**
- Understand the dataset structure (4 variants, 21 sensors)
- Identify low-variance / constant sensors to drop
- Visualize sensor degradation trends
- Compute and visualize the RUL target variable
- Understand operating condition clusters (FD002/FD004)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from preprocessing import (
    load_raw, load_rul_labels, compute_train_rul,
    COLUMNS, INFORMATIVE_SENSORS, DROP_COLS, RUL_CAP
)

DATA_RAW = Path('../data/raw')
FD_IDS = ['FD001', 'FD002', 'FD003', 'FD004']

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('tab10')
print('Setup complete.')

## 1. Load All Datasets

In [ ]:
trains = {fd: load_raw(DATA_RAW, fd, 'train') for fd in FD_IDS}
tests  = {fd: load_raw(DATA_RAW, fd, 'test')  for fd in FD_IDS}

print('\nDataset summary:')
summary = []
for fd in FD_IDS:
    tr = trains[fd]
    te = tests[fd]
    summary.append({
        'Dataset': fd,
        'Train engines': tr['unit_id'].nunique(),
        'Train rows': len(tr),
        'Test engines': te['unit_id'].nunique(),
        'Test rows': len(te),
        'Avg train lifetime': tr.groupby('unit_id')['cycle'].max().mean().round(1),
        'Min train lifetime': tr.groupby('unit_id')['cycle'].max().min(),
    })

pd.DataFrame(summary).set_index('Dataset')

In [ ]:
# Sample of FD001 training data
trains['FD001'].head()

## 2. Sensor Variance Analysis — Which Sensors to Drop?

In [ ]:
def sensor_cv_table(df, fd_id):
    """Coefficient of variation (std/|mean|) per sensor."""
    sensor_cols = [c for c in df.columns if c.startswith('s') or c.startswith('op')]
    stats = df[sensor_cols].agg(['std', 'mean'])
    cv = (stats.loc['std'] / stats.loc['mean'].abs().replace(0, np.nan)).rename('cv')
    rng = (df[sensor_cols].max() - df[sensor_cols].min()).rename('range')
    return pd.DataFrame({'cv': cv, 'range': rng}).sort_values('cv')

cv_fd001 = sensor_cv_table(trains['FD001'], 'FD001')

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['red' if c in DROP_COLS else 'steelblue' for c in cv_fd001.index]
ax.bar(cv_fd001.index, cv_fd001['cv'], color=colors)
ax.axhline(0.01, color='red', linestyle='--', label='Drop threshold (CV < 0.01)')
ax.set_title('Sensor Coefficient of Variation — FD001 (red = dropped)')
ax.set_ylabel('CV')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\nSensors to DROP (zero/near-zero variance):')
print(cv_fd001[cv_fd001.index.isin(DROP_COLS)])

## 3. RUL Distribution — Is the Cap at 125 Appropriate?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, fd_id in zip(axes.flat, FD_IDS):
    df = trains[fd_id]
    lifetimes = df.groupby('unit_id')['cycle'].max()
    ax.hist(lifetimes, bins=30, edgecolor='k', color='steelblue', alpha=0.8)
    ax.axvline(RUL_CAP, color='red', linestyle='--', lw=2, label=f'Cap = {RUL_CAP}')
    ax.set_title(f'{fd_id}  (min={lifetimes.min()}, mean={lifetimes.mean():.0f})')
    ax.set_xlabel('Engine lifetime (cycles)')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Engine Lifetime Distribution per Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f'\nAll min lifetimes > {RUL_CAP}? '
      f'{all(trains[fd].groupby("unit_id")["cycle"].max().min() > RUL_CAP for fd in FD_IDS)}')

## 4. Compute Training RUL (Piecewise-Linear Capped)

In [ ]:
fd001_train = compute_train_rul(trains['FD001'])

# Show the effect of the cap
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Uncapped
max_cycles = fd001_train.groupby('unit_id')['cycle'].max()
fd001_train_temp = fd001_train.copy()
fd001_train_temp['rul_uncapped'] = fd001_train_temp.join(max_cycles.rename('mc'), on='unit_id')['mc'] - fd001_train_temp['cycle']

for uid in [1, 2, 3, 4, 5]:
    sub = fd001_train_temp[fd001_train_temp.unit_id == uid]
    axes[0].plot(sub.cycle, sub.rul_uncapped, alpha=0.7)
    axes[1].plot(sub.cycle, sub.rul, alpha=0.7)

axes[0].set_title('Uncapped RUL (raw)')   
axes[1].set_title(f'Capped RUL (cap={RUL_CAP})')
for ax in axes:
    ax.set_xlabel('Cycle')
    ax.set_ylabel('RUL')
    ax.axhline(RUL_CAP, color='red', linestyle='--', lw=1)

plt.tight_layout()
plt.show()

## 5. Sensor Degradation Trends

In [ ]:
# Show top 6 most RUL-correlated sensors for FD001
corr_with_rul = fd001_train[INFORMATIVE_SENSORS + ['rul']].corr()['rul'].drop('rul').abs().sort_values(ascending=False)
top6_sensors = corr_with_rul.head(6).index.tolist()
print('Top 6 sensors by |correlation with RUL|:')
print(corr_with_rul.head(8).round(3))

In [ ]:
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    has_lowess = True
except ImportError:
    has_lowess = False
    print('statsmodels not installed; skipping LOWESS smoothing')

sample_engines = [1, 2, 3, 4, 5]
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for ax, sensor in zip(axes.flat, top6_sensors):
    for uid in sample_engines:
        sub = fd001_train[fd001_train.unit_id == uid].sort_values('cycle')
        ax.plot(sub.cycle, sub[sensor], alpha=0.35, lw=0.8)
        if has_lowess:
            sm = lowess(sub[sensor].values, sub.cycle.values, frac=0.25)
            ax.plot(sm[:, 0], sm[:, 1], lw=2)
    ax.set_title(sensor)
    ax.set_xlabel('Cycle')

plt.suptitle('Sensor Degradation Trends — FD001 (5 engines)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr_cols = INFORMATIVE_SENSORS + ['rul']
corr = fd001_train[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Sensor Correlation Heatmap — FD001 Training Data')
plt.tight_layout()
plt.show()

## 7. Operating Condition Clustering (FD002 / FD004)

In [ ]:
from sklearn.cluster import KMeans

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, fd_id in zip(axes, ['FD002', 'FD004']):
    df = trains[fd_id]
    km = KMeans(n_clusters=6, random_state=42, n_init=10)
    labels = km.fit_predict(df[['op1', 'op2']].values)
    scatter = ax.scatter(df['op1'], df['op2'], c=labels, cmap='tab10', alpha=0.2, s=3)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', s=200, marker='X', label='Centroids', zorder=5)
    ax.set_title(f'{fd_id} — Operating Condition Clusters')
    ax.set_xlabel('op1 (altitude / Mach)')
    ax.set_ylabel('op2 (TRA)')
    ax.legend()

plt.tight_layout()
plt.show()
print('6 clean clusters visible — per-condition normalization is critical for FD002/FD004')

## 8. Summary

**Sensors to DROP** (zero/near-zero variance across all 4 datasets):
`s1, s5, s6, s10, s16, s18, s19, op3`

**Kept sensors (14):** `s2, s3, s4, s7, s8, s9, s11, s12, s13, s14, s15, s17, s20, s21`

**RUL Cap:** 125 cycles (all engines have minimum lifetime > 128 cycles)

**Normalization:**
- FD001/FD003: global MinMaxScaler
- FD002/FD004: **per-condition** MinMaxScaler (6 clusters on op1/op2)

→ Proceed to `02_feature_engineering.ipynb`